# 02 – Storm Track Analysis

Compute the storm centre (lat, lon, min MSLP) of Hurricane Kirk as forecast
by each OIFS run using Ribberink's `storm_tracker` function.

**Method**: Minimum MSLP within a 4° search box, propagating forward from each
timestep's previous position (initialised from IBTrACS at the first overlap time).

**Reference**: Ribberink et al. (2025), `hurricane_functions.py: storm_tracker`

In [ ]:
import sys, os
sys.path.insert(0, '../scripts')
sys.path.insert(0, '../ribberink_code')

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from oifs_adapter import RUNS, OIFSRun, INIT_DATETIME
import hurricane_functions as hf

## 1. Load IBTrACS best track for Kirk 2024

In [ ]:
# Change to data directory so hf.import_ibtracs finds the file
os.chdir('../data')
best_track = hf.import_ibtracs(hurr_year=2024, storm_name=b'KIRK')
os.chdir('../notebooks')

print('IBTrACS best track:')
print(best_track)
print(f'\nTime range: {best_track.time.values[0]} → {best_track.time.values[-1]}')
print(f'Lat range:  {float(best_track.lat.min()):.1f} to {float(best_track.lat.max()):.1f}')
print(f'Lon range:  {float(best_track.lon.min()):.1f} to {float(best_track.lon.max()):.1f}')

## 2. Run storm tracker for each OIFS experiment

In [ ]:
tracks = {}

for run_name, run_dir in RUNS.items():
    if not os.path.isdir(run_dir):
        print(f'Skipping {run_name}: not found')
        continue
    print(f'Tracking {run_name}...')
    oifs_run = OIFSRun(run_dir)
    msl_ds = oifs_run.get_msl()
    track_df = hf.storm_tracker(msl_ds, best_track, model='OPER', s_size=4)
    tracks[run_name] = track_df
    print(f'  → {len(track_df)} timesteps')

## 3. Save tracks to NetCDF

In [ ]:
os.makedirs('../plots/tracks', exist_ok=True)

for run_name, track_df in tracks.items():
    safe_name = run_name.replace('+', 'p')
    out_path = f'../plots/tracks/track_{safe_name}.nc'
    track_xr = xr.Dataset.from_dataframe(
        track_df.set_index('time') if 'time' in track_df.columns
        else track_df.set_index('datetime')
    )
    track_xr.attrs['run'] = run_name
    track_xr.to_netcdf(out_path)
    print(f'Saved: {out_path}')

## 4. Plot tracks

In [ ]:
colors = {'baserun': '#1f77b4', '+3K': '#d62728', '+6K': '#ff7f0e', '-3K': '#2ca02c'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: map track
ax = axes[0]
for run_name, track_df in tracks.items():
    ax.plot(track_df['lon'], track_df['lat'], '.-',
            color=colors.get(run_name, 'k'), label=run_name)

# IBTrACS best track for reference
ax.plot(best_track.lon, best_track.lat, 'k--', lw=2, label='IBTrACS')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Hurricane Kirk 2024 – OIFS Track')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: min MSLP time series
ax2 = axes[1]
for run_name, track_df in tracks.items():
    time_col = 'time' if 'time' in track_df.columns else 'datetime'
    mslp_col = 'msl' if 'msl' in track_df.columns else track_df.columns[-1]
    ax2.plot(track_df[time_col], track_df[mslp_col] / 100,  # Pa → hPa
             '.-', color=colors.get(run_name, 'k'), label=run_name)

ax2.set_xlabel('Date')
ax2.set_ylabel('Min MSLP (hPa)')
ax2.set_title('Kirk – Minimum Sea-Level Pressure')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30)

plt.tight_layout()
plt.savefig('../plots/kirk_tracks.png', dpi=150)
plt.show()
print('Saved: ../plots/kirk_tracks.png')